# Online Purchase Intention Prediction
## Data Methodology and Logistic Regression



## Analysis code

Run the next cell once. It generates the EDA graphics, CV result tables, held-out predictions, coefficient table and `analysis_summary.json` in a new `lr_results` folder beside the notebook. 


In [1]:

from __future__ import annotations


import warnings
warnings.filterwarnings("ignore", category=FutureWarning, message="Setting an item of incompatible dtype.*")
warnings.filterwarnings("ignore", category=UserWarning, message="Found unknown categories.*")

import json
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    balanced_accuracy_score,
    ConfusionMatrixDisplay,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    RocCurveDisplay,
    PrecisionRecallDisplay,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler


SEED = 6805

# Keep this notebook and the two CSV files in the same folder. In Jupyter,
# Path.cwd() is the notebook's working folder, so no user-specific path is needed.
DATA_DIR = Path.cwd().resolve()
TRAIN_PATH = DATA_DIR / "online_shoppers_train_fixed.csv"
TEST_PATH = DATA_DIR / "online_shoppers_test_fixed.csv"
RESULTS_DIR = DATA_DIR / "lr_results"
FIGURES_DIR = RESULTS_DIR / "figures"

missing_files = [path.name for path in (TRAIN_PATH, TEST_PATH) if not path.exists()]
if missing_files:
    raise FileNotFoundError(
        "Please place the notebook and these CSV files in the same folder: "
        + ", ".join(missing_files)
    )

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

TARGET = "Revenue"
NUMERIC_FEATURES = [
    "Administrative", "Administrative_Duration", "Informational", "Informational_Duration",
    "ProductRelated", "ProductRelated_Duration", "BounceRates", "ExitRates", "PageValues",
    "SpecialDay",
]
CATEGORICAL_FEATURES = [
    "Month", "OperatingSystems", "Browser", "Region", "TrafficType", "VisitorType", "Weekend",
]
SKEWED_NONNEGATIVE = [
    "Administrative", "Administrative_Duration", "Informational", "Informational_Duration",
    "ProductRelated", "ProductRelated_Duration", "BounceRates", "ExitRates", "SpecialDay",
]

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({"figure.dpi": 140, "savefig.dpi": 220, "axes.titleweight": "bold"})


def clean_target(series: pd.Series) -> pd.Series:
    """Normalise the binary target without assuming the source representation."""
    if series.dtype == bool:
        return series.astype(int)
    return series.astype(str).str.strip().str.lower().map({"true": 1, "false": 0, "1": 1, "0": 0}).astype(int)


def apply_log1p(frame: pd.DataFrame) -> pd.DataFrame:
    """Apply log1p only to non-negative, long-tailed numeric fields."""
    out = frame.copy()
    columns = [col for col in SKEWED_NONNEGATIVE if col in out.columns]
    out.loc[:, columns] = np.log1p(out.loc[:, columns].astype(float))
    return out


def make_preprocessor(include_pagevalues: bool, log_transform: bool = True) -> tuple[ColumnTransformer, list[str], list[str]]:
    numeric = [col for col in NUMERIC_FEATURES if include_pagevalues or col != "PageValues"]
    categoricals = CATEGORICAL_FEATURES.copy()
    steps = []
    if log_transform:
        steps.append(("log1p", FunctionTransformer(apply_log1p, validate=False, feature_names_out="one-to-one")))
    steps.append(("scale", StandardScaler()))
    numeric_pipe = Pipeline(steps)
    preprocessor = ColumnTransformer(
        transformers=[
            ("numeric", numeric_pipe, numeric),
            ("categorical", OneHotEncoder(handle_unknown="ignore", drop="if_binary"), categoricals),
        ],
        remainder="drop",
        verbose_feature_names_out=False,
    )
    return preprocessor, numeric, categoricals


def make_pipeline(include_pagevalues: bool, log_transform: bool = True) -> Pipeline:
    preprocessor, _, _ = make_preprocessor(include_pagevalues, log_transform)
    return Pipeline(
        steps=[
            ("preprocess", preprocessor),
            ("model", LogisticRegression(
                penalty="l2", solver="liblinear", class_weight="balanced", max_iter=5000, random_state=SEED
            )),
        ]
    )


def choose_threshold(y_true: pd.Series, probabilities: np.ndarray) -> float:
    precision, recall, thresholds = precision_recall_curve(y_true, probabilities)
    f1_values = 2 * precision * recall / np.maximum(precision + recall, 1e-12)
    # Last precision/recall point has no corresponding threshold.
    best_idx = int(np.nanargmax(f1_values[:-1]))
    return float(thresholds[best_idx])


def evaluate(y_true: pd.Series, probabilities: np.ndarray, threshold: float) -> dict:
    predictions = (probabilities >= threshold).astype(int)
    return {
        "threshold": float(threshold),
        "roc_auc": float(roc_auc_score(y_true, probabilities)),
        "pr_auc": float(average_precision_score(y_true, probabilities)),
        "precision": float(precision_score(y_true, predictions, zero_division=0)),
        "recall": float(recall_score(y_true, predictions, zero_division=0)),
        "f1": float(f1_score(y_true, predictions, zero_division=0)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, predictions)),
        "tn": int(confusion_matrix(y_true, predictions)[0, 0]),
        "fp": int(confusion_matrix(y_true, predictions)[0, 1]),
        "fn": int(confusion_matrix(y_true, predictions)[1, 0]),
        "tp": int(confusion_matrix(y_true, predictions)[1, 1]),
    }


def vif_table(frame: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    # VIF is calculated on complete numerical training data. This is diagnostic only, not a selection rule.
    x = frame[columns].astype(float).copy()
    x = pd.DataFrame(StandardScaler().fit_transform(x), columns=columns)
    values = []
    for column in columns:
        others = [other for other in columns if other != column]
        r2 = LinearRegression().fit(x[others], x[column]).score(x[others], x[column])
        values.append({"feature": column, "vif": float(1 / max(1 - r2, 1e-10))})
    return pd.DataFrame(values).sort_values("vif", ascending=False, ignore_index=True)


def build_eda(train: pd.DataFrame, y_train: pd.Series, numeric_no_pagevalues: list[str]) -> dict:
    summary = {}
    class_counts = y_train.value_counts().sort_index()
    summary["train_positive_rate"] = float(y_train.mean())
    summary["train_class_counts"] = {str(key): int(value) for key, value in class_counts.items()}
    summary["pagevalues_zero_share"] = float((train["PageValues"] == 0).mean())
    summary["purchase_rate_pagevalues_zero"] = float(y_train.loc[train["PageValues"] == 0].mean())
    summary["purchase_rate_pagevalues_positive"] = float(y_train.loc[train["PageValues"] > 0].mean())

    fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
    labels = ["No purchase", "Purchase"]
    axes[0].bar(labels, class_counts.values, color=["#95a5a6", "#d35400"])
    axes[0].set_title("Training outcome distribution")
    axes[0].set_ylabel("Sessions")
    for idx, count in enumerate(class_counts.values):
        axes[0].text(idx, count + class_counts.max() * 0.02, f"{count:,}\n({count / len(y_train):.1%})", ha="center")
    pv_group = pd.DataFrame({"Group": np.where(train["PageValues"] > 0, "PageValues > 0", "PageValues = 0"), "Revenue": y_train})
    rates = pv_group.groupby("Group", observed=True)["Revenue"].mean().reindex(["PageValues = 0", "PageValues > 0"])
    axes[1].bar(rates.index, rates.values, color=["#7f8c8d", "#2980b9"])
    axes[1].set_title("Purchase rate by PageValues")
    axes[1].set_ylabel("Purchase rate")
    axes[1].set_ylim(0, min(1, rates.max() * 1.28))
    for idx, value in enumerate(rates.values):
        axes[1].text(idx, value + 0.01, f"{value:.1%}", ha="center")
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / "eda_class_balance_pagevalues.png", bbox_inches="tight")
    plt.close(fig)

    corr_columns = numeric_no_pagevalues
    correlation = train[corr_columns].corr(method="spearman")
    fig, ax = plt.subplots(figsize=(9.2, 7.5))
    sns.heatmap(correlation, cmap="vlag", center=0, annot=True, fmt=".2f", square=True, ax=ax, cbar_kws={"shrink": 0.75})
    ax.set_title("Spearman correlations of numeric predictors in training data")
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / "eda_numeric_spearman_heatmap.png", bbox_inches="tight")
    plt.close(fig)

    month_rates = pd.DataFrame({"Month": train["Month"].astype(str), "Revenue": y_train}).groupby("Month", observed=True)["Revenue"].agg(["mean", "size"])
    month_order = ["Feb", "Mar", "May", "June", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
    month_rates = month_rates.reindex([month for month in month_order if month in month_rates.index])
    fig, ax = plt.subplots(figsize=(9, 4.5))
    ax.bar(month_rates.index, month_rates["mean"], color="#8e44ad")
    ax.set_title("Training purchase rate by month")
    ax.set_ylabel("Purchase rate")
    ax.set_ylim(0, min(1, month_rates["mean"].max() * 1.25))
    for idx, value in enumerate(month_rates["mean"]):
        ax.text(idx, value + 0.003, f"{value:.1%}", ha="center", fontsize=9)
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / "eda_purchase_rate_month.png", bbox_inches="tight")
    plt.close(fig)
    return summary


def fit_specification(name: str, include_pagevalues: bool, x_train: pd.DataFrame, y_train: pd.Series, x_test: pd.DataFrame, y_test: pd.Series, cv: StratifiedKFold, log_transform: bool = True) -> tuple[dict, Pipeline]:
    pipeline = make_pipeline(include_pagevalues=include_pagevalues, log_transform=log_transform)
    grid = GridSearchCV(
        estimator=pipeline,
        param_grid={"model__C": [0.01, 0.1, 1.0, 10.0, 100.0]},
        scoring={"pr_auc": "average_precision", "roc_auc": "roc_auc", "f1": "f1", "recall": "recall"},
        refit="pr_auc",
        cv=cv,
        n_jobs=1,
        return_train_score=False,
    )
    grid.fit(x_train, y_train)
    best_estimator = grid.best_estimator_
    oof_probabilities = cross_val_predict(clone(best_estimator), x_train, y_train, cv=cv, method="predict_proba", n_jobs=1)[:, 1]
    cv_threshold = choose_threshold(y_train, oof_probabilities)
    test_probabilities = best_estimator.predict_proba(x_test)[:, 1]
    default_metrics = evaluate(y_test, test_probabilities, threshold=0.50)
    selected_metrics = evaluate(y_test, test_probabilities, threshold=cv_threshold)
    results_frame = pd.DataFrame(grid.cv_results_)
    columns = ["param_model__C", "mean_test_pr_auc", "std_test_pr_auc", "mean_test_roc_auc", "mean_test_f1", "mean_test_recall", "rank_test_pr_auc"]
    cv_table = results_frame[columns].sort_values("rank_test_pr_auc").copy()
    cv_table.columns = ["C", "CV PR AUC mean", "CV PR AUC sd", "CV ROC AUC mean", "CV F1 mean", "CV recall mean", "rank"]
    cv_table.to_csv(RESULTS_DIR / f"cv_results_{name}.csv", index=False)
    return {
        "name": name,
        "include_pagevalues": include_pagevalues,
        "log_transform": log_transform,
        "best_C": float(grid.best_params_["model__C"]),
        "best_cv_pr_auc": float(grid.best_score_),
        "cv_table": cv_table.to_dict(orient="records"),
        "threshold_from_oof_f1": cv_threshold,
        "test_default_threshold": default_metrics,
        "test_cv_selected_threshold": selected_metrics,
        "test_probabilities": test_probabilities.tolist(),
    }, best_estimator


def coefficient_table(estimator: Pipeline) -> pd.DataFrame:
    names = estimator.named_steps["preprocess"].get_feature_names_out()
    coefficients = estimator.named_steps["model"].coef_.ravel()
    table = pd.DataFrame({"feature": names, "coefficient": coefficients})
    table["odds_ratio"] = np.exp(table["coefficient"])
    return table.sort_values("coefficient", ascending=False, ignore_index=True)


def make_model_figures(y_test: pd.Series, main_result: dict, sensitivity_result: dict) -> None:
    main_prob = np.array(main_result["test_probabilities"])
    sens_prob = np.array(sensitivity_result["test_probabilities"])
    fig, axes = plt.subplots(1, 2, figsize=(11.2, 4.4))
    RocCurveDisplay.from_predictions(y_test, main_prob, name="Exclude PageValues", ax=axes[0], color="#d35400")
    RocCurveDisplay.from_predictions(y_test, sens_prob, name="Include PageValues", ax=axes[0], color="#2980b9")
    axes[0].set_title("Held-out test ROC curves")
    PrecisionRecallDisplay.from_predictions(y_test, main_prob, name="Exclude PageValues", ax=axes[1], color="#d35400")
    PrecisionRecallDisplay.from_predictions(y_test, sens_prob, name="Include PageValues", ax=axes[1], color="#2980b9")
    axes[1].axhline(y_test.mean(), linestyle="--", color="gray", label=f"Baseline = {y_test.mean():.1%}")
    axes[1].set_title("Held-out test precision-recall curves")
    axes[1].legend(loc="lower left")
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / "test_roc_pr_curves.png", bbox_inches="tight")
    plt.close(fig)

    threshold = main_result["threshold_from_oof_f1"]
    cm = confusion_matrix(y_test, (main_prob >= threshold).astype(int))
    fig, ax = plt.subplots(figsize=(5.2, 4.5))
    ConfusionMatrixDisplay(cm, display_labels=["No purchase", "Purchase"]).plot(ax=ax, colorbar=False, cmap="Oranges")
    ax.set_title(f"Main LR test confusion matrix\nThreshold selected from 5-fold OOF predictions ({threshold:.2f})")
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / "test_confusion_matrix_main.png", bbox_inches="tight")
    plt.close(fig)


def main() -> None:
    train_raw = pd.read_csv(TRAIN_PATH)
    test_raw = pd.read_csv(TEST_PATH)
    if set(train_raw.columns) != set(test_raw.columns):
        raise ValueError("Train and test columns differ.")
    if TARGET not in train_raw.columns:
        raise ValueError(f"Expected target column {TARGET!r} was not found.")

    y_train = clean_target(train_raw[TARGET])
    y_test = clean_target(test_raw[TARGET])
    train = train_raw.drop(columns=TARGET).copy()
    test = test_raw.drop(columns=TARGET).copy()

    duplicated_train = int(train_raw.duplicated().sum())
    duplicated_test = int(test_raw.duplicated().sum())
    # Test observations remain untouched because the supplied split is the held-out evaluation set.
    train = train.loc[~train_raw.duplicated()].reset_index(drop=True)
    y_train = y_train.loc[~train_raw.duplicated()].reset_index(drop=True)
    combined_duplicate_rows = int(pd.concat([train_raw, test_raw], ignore_index=True).duplicated().sum())
    missing_train = int(train.isna().sum().sum())
    missing_test = int(test.isna().sum().sum())
    negative_numeric = {column: int((train[column] < 0).sum()) for column in NUMERIC_FEATURES}

    preprocessor, numeric_no_pagevalues, categorical_features = make_preprocessor(include_pagevalues=False)
    eda_summary = build_eda(train, y_train, numeric_no_pagevalues)
    vif = vif_table(train, numeric_no_pagevalues)
    vif.to_csv(RESULTS_DIR / "numeric_vif_main_specification.csv", index=False)

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    main_result, main_estimator = fit_specification("exclude_pagevalues", False, train, y_train, test, y_test, cv, log_transform=True)
    sensitivity_result, sensitivity_estimator = fit_specification("include_pagevalues", True, train, y_train, test, y_test, cv, log_transform=True)
    raw_result, _ = fit_specification("exclude_pagevalues_raw_numeric", False, train, y_train, test, y_test, cv, log_transform=False)

    coefficients = coefficient_table(main_estimator)
    coefficients.to_csv(RESULTS_DIR / "main_model_coefficients_odds_ratios.csv", index=False)
    make_model_figures(y_test, main_result, sensitivity_result)

    # Keep the final JSON compact and make probabilities available in a separate test-prediction file.
    pd.DataFrame({
        "actual_revenue": y_test,
        "probability_excluding_pagevalues": main_result.pop("test_probabilities"),
        "probability_including_pagevalues": sensitivity_result.pop("test_probabilities"),
        "probability_raw_numeric": raw_result.pop("test_probabilities"),
    }).to_csv(RESULTS_DIR / "heldout_test_predictions.csv", index=False)

    report = {
        "seed": SEED,
        "data_quality": {
            "raw_train_rows": int(len(train_raw)),
            "raw_test_rows": int(len(test_raw)),
            "training_rows_after_within_train_deduplication": int(len(train)),
            "duplicate_rows_in_raw_train": duplicated_train,
            "duplicate_rows_in_heldout_test": duplicated_test,
            "duplicate_rows_in_combined_supplied_files": combined_duplicate_rows,
            "missing_cells_train": missing_train,
            "missing_cells_test": missing_test,
            "negative_numeric_counts_train": negative_numeric,
            "categorical_features": categorical_features,
            "numeric_features_main_specification": numeric_no_pagevalues,
        },
        "eda": eda_summary,
        "vif_top": vif.head(10).round(4).to_dict(orient="records"),
        "main_excluding_pagevalues": main_result,
        "sensitivity_including_pagevalues": sensitivity_result,
        "sensitivity_raw_numeric": raw_result,
        "main_top_positive_coefficients": coefficients.head(10).round(5).to_dict(orient="records"),
        "main_top_negative_coefficients": coefficients.tail(10).sort_values("coefficient").round(5).to_dict(orient="records"),
    }
    with (RESULTS_DIR / "analysis_summary.json").open("w", encoding="utf-8") as handle:
        json.dump(report, handle, ensure_ascii=False, indent=2)

    print(json.dumps({
        "main_test": main_result["test_cv_selected_threshold"],
        "pagevalues_sensitivity_test": sensitivity_result["test_cv_selected_threshold"],
        "best_C_main": main_result["best_C"],
        "best_cv_pr_auc_main": main_result["best_cv_pr_auc"],
    }, ensure_ascii=False, indent=2))





## Execute the pipeline

The main specification excludes `PageValues`; the code also fits a PageValues-included sensitivity model and a raw-numeric transformation sensitivity model.


In [2]:
main()


{
  "main_test": {
    "threshold": 0.5484515482626051,
    "roc_auc": 0.7565610307448591,
    "pr_auc": 0.3354329659412846,
    "precision": 0.2952941176470588,
    "recall": 0.6570680628272252,
    "f1": 0.4074675324675325,
    "balanced_accuracy": 0.6830750707530977,
    "tn": 1460,
    "fp": 599,
    "fn": 131,
    "tp": 251
  },
  "pagevalues_sensitivity_test": {
    "threshold": 0.6358958631931299,
    "roc_auc": 0.8998726062822139,
    "pr_auc": 0.6221420238738855,
    "precision": 0.6098130841121495,
    "recall": 0.6832460732984293,
    "f1": 0.6444444444444445,
    "balanced_accuracy": 0.8010693698206571,
    "tn": 1892,
    "fp": 167,
    "fn": 121,
    "tp": 261
  },
  "best_C_main": 0.01,
  "best_cv_pr_auc_main": 0.3401379815104967
}


## Reading the results

- Use PR-AUC alongside ROC-AUC because Revenue is imbalanced.
- Report the PageValues model as a sensitivity analysis unless feature availability prior to conversion is verified.
- Interpret odds ratios as conditional predictive associations, not causal effects.
- The selected operating threshold is obtained from out-of-fold training predictions, not from the held-out test set.
